In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("balance", DecimalType(12,2), True),
    StructField("updated_date", DateType(), True)
])

df = (
    spark.read
         .option("header", "true")
         .schema(schema)
         .csv("/Volumes/banking/bronze/landing_volume/customer_updates_window.csv")
)

display(df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

In [0]:
window_spec = Window.partitionBy("customer_id") \
                    .orderBy(col("updated_date").desc())

In [0]:
df_ranked = df.withColumn(
    "rn",
    row_number().over(window_spec)
)

display(df_ranked)

## keep only the latest record

In [0]:
latest_df = (
    df_ranked
        .filter(col("rn") == 1)
        .drop("rn")
)

display(latest_df)